In [1]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# 01. Fiber Characterization & SSFM Validation\n",
    "Validating the Split-Step Fourier Method (SSFM) for the generalized nonlinear Schrödinger equation (GNLSE) over 80 km of standard single-mode fiber (SSMF).\n",
    "\n",
    "**Target Parameters:** 80 km SSMF, 42.4 THz aggregate bandwidth (O+E+S+C+L).\n",
    "\n",
    "**Goal:** Prove numerical convergence and validate Raman cross-band interactions."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import sys\n",
    "sys.path.append('..')\n",
    "import yaml\n",
    "import numpy as np\n",
    "import matplotlib.pyplot as plt\n",
    "from src.modulation import build_band_plan, generate_pcs_symbols\n",
    "from src.fiber_channel import run_ssfm\n",
    "\n",
    "with open('../config/simulation_params.yaml', 'r') as f:\n",
    "    cfg = yaml.safe_load(f)\n",
    "\n",
    "print(f\"Fiber Length: {cfg['fiber']['length_km']} km\")\n",
    "print(f\"Attenuation: {cfg['fiber']['attenuation_dB_per_km']} dB/km\")\n",
    "print(f"Beta2: {cfg['fiber']['beta2_s2_per_m']} s^2/m\")\n",
    "print(f"Gamma: {cfg['fiber']['gamma_per_W_per_m']} 1/W/m\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 1.1 SSFM Convergence Test\n",
    "Testing the symmetric split-step method with a single-channel pulse to verify dispersion and nonlinearity balance."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Build a simplified single-band plan for quick testing\n",
    "test_bands_cfg = {'C': cfg['bands']['C']}\n",
    "test_agg_cfg = {'total_channels': 1, 'total_bandwidth_THz': 0.1}\n",
    "test_bands_cfg['C']['channels'] = 1\n",
    "\n",
    "band_plan = build_band_plan(test_bands_cfg, test_agg_cfg)\n",
    "tx = generate_pcs_symbols(band_plan, cfg['modulation'], cfg['launch_power'], samples=4096)\n",
    "\n",
    "# Run SSFM\n",
    "rx_raw = run_ssfm(tx, cfg['fiber'], cfg['ssfm'])\n",
    "\n",
    "# Plot input vs output spectrum\n",
    "plt.figure(figsize=(10, 5))\n",
    "plt.plot(np.abs(tx['field'][0, :]), label='Tx Field', alpha=0.7)\n",
    "plt.plot(np.abs(rx_raw['field'][0, :]), label='Rx Field (80km)', alpha=0.7)\n",
    "plt.title('SSFM Single-Channel Pulse Propagation')\n",
    "plt.xlabel('Samples')\n",
    "plt.ylabel('Amplitude')\n",
    "plt.legend()\n",
    "plt.grid(True)\n",
    "plt.show()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 1.2 Raman Cross-Band Interaction Analysis\n",
    "Observing power transfer from higher frequencies (L-band) to lower frequencies (O-band) over 80 km."
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "band_plan_full = build_band_plan(cfg['bands'], cfg['aggregate'])\n",
    "tx_full = generate_pcs_symbols(band_plan_full, cfg['modulation'], cfg['launch_power'], samples=8192)\n",
    "\n",
    "# Run SSFM with Raman enabled\n",
    "rx_full = run_ssfm(tx_full, cfg['fiber'], {**cfg['ssfm'], 'include_raman': True})\n",
    "\n",
    "# Plot aggregate spectrum\n",
    "psd_tx = np.abs(np.fft.fftshift(np.fft.fft(tx_full['field'], axis=1)))**2\n",
    "psd_rx = np.abs(np.fft.fftshift(np.fft.fft(rx_full['field'], axis=1)))**2\n",
    "\n",
    "plt.figure(figsize=(12, 5))\n",
    "plt.plot(10*np.log10(np.mean(psd_tx, axis=0)+1e-12), label='Tx Spectrum')\n",
    "plt.plot(10*np.log10(np.mean(psd_rx, axis=0)+1e-12), label='Rx Spectrum (with Raman)')\n",
    "plt.title('O+E+S+C+L Spectrum Before and After 80 km SSMF')\n",
    "plt.xlabel('Frequency Bins')\n",
    "plt.ylabel('Power Spectral Density (dB)')\n",
    "plt.legend()\n",
    "plt.grid(True)\n",
    "plt.show()"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "codemirror_mode": {
    "name": "ipython",
    "version": 3
   },
   "file_extension": ".py",
   "mimetype": "text/x-python",
   "name": "python",
   "nbconvert_exporter": "python",
   "pygments_lexer": "ipython3",
   "version": "3.8.10"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 5
}

SyntaxError: '[' was never closed (3285485502.py, line 20)